<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items
The aim is to detect similar textual items in the `text` field of the Kaggle [Yelp](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset) dataset.

We first of all import the Yelp dataset from Kaggle, using a token.

In [1]:
import os
import json
import pandas as pd
import pip
import string

os.environ['KAGGLE_USERNAME'] = "luciaannamellini"
os.environ['KAGGLE_KEY'] = "c209fcf223ecdd6be8fd373196354f4b"
!kaggle datasets download -d yelp-dataset/yelp-dataset

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


yelp-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [2]:
from tqdm import tqdm
import zipfile

DATA_DIR = "/content/yelp-dataset"

with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
     for file in tqdm(iterable=zip_ref.namelist(), total=len(zip_ref.namelist())):
          zip_ref.extract(member=file, path=DATA_DIR)

100%|██████████| 6/6 [02:22<00:00, 23.76s/it]


We now prepare the entry point for the Spark functionalities that will we used later on.

In [3]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init("spark-3.5.0-bin-hadoop3")# SPARK_HOME
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We take into consideration the portion of the dataset regarding reviews, contained in `yelp_academic_dataset_review.json`. We convert the resulting dataframe in RDD form, seen that from now on we will manage data in this format.

In [4]:
df_reviews = spark.read.json("/content/yelp-dataset/yelp_academic_dataset_review.json")
reviews_RDD = df_reviews.rdd

We sample the dataset at the only scope to speed up the following steps in the (free) Google Colab environment.

In [5]:
reviews_RDD = reviews_RDD.sample(False, 0.00001, 42)
reviews_num = reviews_RDD.count()
print("The chosen random sample contains {} reviews".format(reviews_num))

The chosen random sample contains 66 reviews


Seen the aim of the project we will only be looking at the `text` attribute of the imported reviews. From now on we will refer to the `text` field of a review, simply as *review*.

In [6]:
reviews_RDD = reviews_RDD.map((lambda r: (r[0], r['text'])))

To have a better idea, let's look at some reviews from the dataset.

In [7]:
print('---\n')
for text in reviews_RDD.take(3):
    print('{}\n'.format(text[1]))
    print('---\n')

---

This annual June event celebrating Indian jewelry, art and food turns out large crowds. First, the good: Lots of jewelry vendors offer tons of variety, from Santa Fe and New Mexico flavored pieces to bright, chunky tribal ones. Necklaces, door chains, bracelets, earrings, even beaded purses and more can be found here. One unusual piece that was just out of my price range (at $500!) was a gorgeous eagle carved out of bone or ivory on a circular metal necklace. 

Now the bad: The Eiteljorg's parking lot is not open to festivalgoers, despite the fact that they're the ones hosting the event and it's a $10 charge to enter. Further, the food offerings are more Hoosier than Hopi--the closest thing to Native American cuisine here was the beef jerky sold in the gift shop. I'm joking--a little. 

Indian Fry Tacos were sold, but they make use of flour and lard, ingredients introduced to Native Americans via the U.S. government following a forced expulsion during the Trail of Tears. One Mexic

## Data pre-processing

We begin by removing text cells that are `None` or that contain empty strings. It is possible to verify that each review is nonempty.

In [8]:
reviews_RDD = reviews_RDD.filter(lambda text: bool(text))

To study the similarity between reviews we look at the corresponding strings as sets of tokens. We have chosen to divide each review in the terms composing it, in the lower case version. We have preferred this approach against using classical $k$-grams because we are more intersted in the meaning of the reviews, so we avoid looking also at the structure of the text.

We get rid of the stop words appearing in the tokens to extract the actual semantics of the text. Also, we lemmatize the remaining tokens: we are oblivious about the various inflections of a certain word. Lastly, for each review we maintain each token once.

In [9]:
%%capture
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = stopwords.words('english')
import spacy
nlp = spacy.load("en_core_web_sm")

split_regex = r'\W+'

tokenize = lambda string: [s for s in re.split(split_regex,string.lower()) if s not in stopwords and s != '']
review_tokens_RDD = reviews_RDD.map(lambda s: (s[0],tokenize(s[1])))

lemmatize = lambda word: str([token.lemma_ for token in nlp(word)][0])
tokens_in_review_RDD = review_tokens_RDD.map(lambda s: (s[0], [lemmatize(w) for w in s[1]]))

tokens_in_review_RDD = tokens_in_review_RDD.map(lambda s: (s[0], list(set(s[1]))))

Let's look at how the first of the reviews that we showed above has been transformed.

In [10]:
tokens_in_review_RDD.first()

('4ieBZ94-Bpn7BBvN1fGCyQ',
 ['shop',
  'little',
  'ice',
  'vendor',
  'find',
  'charge',
  'eiteljorg',
  'enter',
  'state',
  'cuisine',
  'flour',
  'tamale',
  'joke',
  'chunky',
  'price',
  'even',
  'lot',
  'door',
  'mexico',
  'turn',
  'parking',
  'open',
  'ton',
  'chain',
  'authentic',
  'new',
  'close',
  'cream',
  'delicious',
  'bead',
  'bright',
  'bone',
  'one',
  'range',
  'force',
  'bad',
  'make',
  'festivalgoer',
  'variety',
  'offering',
  'fry',
  'unusual',
  'jewelry',
  'ingredient',
  'regular',
  'gorgeous',
  'hope',
  'use',
  'purse',
  'american',
  'tacos',
  'via',
  'eagle',
  'root',
  'june',
  'hopi',
  'beef',
  'large',
  'first',
  'crowd',
  'americans',
  'follow',
  'jerky',
  'carve',
  '10',
  'gift',
  'trail',
  'piece',
  'indian',
  'u',
  'native',
  'flavor',
  'hoosier',
  'tear',
  '500',
  'fe',
  'tribal',
  'fact',
  'celebrate',
  'art',
  'thing',
  'despite',
  'food',
  'expulsion',
  'santa',
  'government',


At this point we have codified each review through it's essential information.

For later steps we also prepare the transpose of this point of view, by listing for each token the reviews to which it belongs to.

In [11]:
reviews_for_token_RDD = tokens_in_review_RDD.flatMap(lambda s: [(t,s[0]) for t in s[1]]).groupByKey().mapValues(list)

## Similar items with Jaccard similarity

We begin by evaluating the similarity of the reviews according to the Jaccard similarity measure between sets. The Jaccard similarity between sets $S$ and $T$ is defined as:
\begin{equation}
J(S,T)=\frac{|S\cap T|}{|S\cup T|}
\end{equation}
In our case the items of the sets are the tokens that we have extracted from the reviews, so the sets are the summaries of the reviews.

### Similarity preserving summary of the reviews
To store the tokens extracted from each review compactly we consider their characteristic matrix, used to represent a collection of sets. Such a structure is defined as a binary matrix, which holds all the reviews in the columns, and all possible tokens in the rows. A cell is set to $1$ only if the corresponding token appears in the indicated review, and $0$ otherwise.

Though with growing amounts of data it is not realistic to maintain the whole characteristic matrix. So, we construct a succint structure called a signature matrix that scales down the number of rows.    
Each row of such matrix is constructed as follows:
1. choose a permutation of the rows of the characteristic matrix uniformly at random among all possible permutations,
2. apply the chosen permutation to the rows of the charactistic matrix,
3. apply a minhash function to all columns of the resulting matrix.

A minhash function is defined as $h:\{\text{reviews}\}\to\{\text{tokens}\}$, and for a review it returns the index of the first token appearing in the column of the characteristic matrix.

The column related to a review in the signature matrix is it's signature. This signature depends on the random permutations applied to the rows of the chacteristic matrix.

We begin by putting aside all the *distinct* tokens found in the dataset, and their amount.

In [12]:
all_tokens_RDD = tokens_in_review_RDD.flatMap(lambda t: [(s, 1) for s in t[1]]).reduceByKey(lambda a,b: a+b).map(lambda s: (s[0],1))
all_tokens_num = all_tokens_RDD.count()

The length of the signatures can be adjusted by setting the variable `signature_length`.

In [65]:
signature_length = 200

To build the signature matrix we start by preparing all the permutations of the rows of the characteristic matrix, in number equal to the length of the signatures. Here, in `token_permutations_RDD` for each token we store the position that it occupies in each permutation.  
  

In [ ]:
import numpy as np
permutations_RDD = sc.parallelize(range(signature_length)).map(lambda s: (s,list(np.random.permutation(all_tokens_num))))
all_tokens_index_RDD = all_tokens_RDD.map(lambda s: s[0]).zipWithIndex()
token_permutations_RDD = all_tokens_index_RDD.cartesian(permutations_RDD).map(lambda s: (s[0][0],(s[1][0],s[1][1][s[0][1]]))).groupByKey().mapValues(list)

At this point we join the reviews, the relative tokens and the indices assumed by the tokens in each of the permutations.

In [ ]:
token_review_indices_RDD = token_permutations_RDD.join(reviews_for_token_RDD)

We transform this RDD to arrive at a point in which each tuple is in the form $((r_i, p_j),[i_1,i_2,\dots,i_k])$, where:
* $r_j$ is a generic review,
* $p_j$ is one of the permutations fixed above,
* $[i_1,i_2,\dots,i_k]$ is a list of the indices corresponding to the tokens contained in $r_j$ according to the order dictated by $p_j$.

In [ ]:
token_index_review_RDD = token_review_indices_RDD.map(lambda s: ((s[0],s[1][0]),s[1][1])).flatMapValues(lambda s: s)
review_indices_RDD = token_index_review_RDD.map(lambda s: (s[1],s[0][1])).flatMapValues(lambda s: s)
review_permutation_indices_RDD = review_indices_RDD.map(lambda s: ((s[0],s[1][0]),s[1][1])).groupByKey().mapValues(list)

From these tuples it is immediate to compute the minhash for each review for all the fixed permutations. Specifically, we have found the signature for each review.

In [ ]:
review_minhash_RDD = review_permutation_indices_RDD.map(lambda s: (s[0],min(s[1])))
review_signature_RDD = review_minhash_RDD.map(lambda s: (s[0][0],s[1])).groupByKey().mapValues(list)

Let's look at the signature for a given review.

In [ ]:
review_signature_RDD.first()

### Locality-sensitive hashing (LSH)

It would be unthinkable to compare all possible pairs of reviews to find similar ones among them. This would mean scanning all the rows in the signature matrix to compute the relative frequency between possible pairs of reviews. So, we proceed by applying Locality-Sensitive Hashing. In this approach we reduce the number of rows that determine the signature of a review by hashing so called bands of rows. The rationale is that similar reviews are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity. Looking at the resulting signatures we consider as a candidate pair only those for which their Jaccard similarity exceeds a threshold $t$.

We begin by dividing the signature matrix into $b$ bands of $r$ rows each. The choice of $r$ and $b$ depends on the threshold $t$ on the Jaccard similarity between pairs of reviews. The value of the threshold $t$ is approximately the value of similarity at which the probability of becoming a candidate is $\frac{1}{2}$.
So, we keep into account the following relationships (for brevity $l$=`signature_length`):   

\begin{equation}
    \begin{cases}
        t=\left(\frac{1}{b}\right)^\frac{1}{r} \\
        r\cdot b = l
    \end{cases}
\end{equation}

The function `band_size` solves this system in $r$ by computing it's value through [^1]
\begin{equation}
    r=-\frac{W(-l\ln(t))}{\ln(t)}
\end{equation}

[^1]:$W$ is the Lambert $W$ function, used to solve equations in the form $we^{w}=z$ for $w$

In [67]:
import math
from scipy.special import lambertw

def band_size(t, sig_len):
    return -math.ceil(lambertw(-sig_len*np.log(t)).real/np.log(t))

t=0.9
r=band_size(t,signature_length)
b=math.ceil(signature_length/r)
print("The chosen parameters are: \n r: {} \n b: {}".format(r,b))

The chosen parameters are: 
 r: 21 
 b: 10


Having chosen the parameters we proceed by subdividing the rows of the similarity matrix. In `review_banded_signature_RDD` this is simply done by storing for each review a sequence of lists, each of which contains the portion of the relative signature assigned to a band.

In [ ]:
def split_list_with_index(l,b):
    split_list = [(list(a)) for a in np.array_split(np.array(l), b)]
    return [split_list[i] for i in range(b)]

review_banded_signature_RDD = review_signature_RDD.map(lambda s: (s[0],split_list_with_index(s[1],b)))

We proceed by hashing for each token the rows in each of the $b$. For each band we use a different hashing function, to avoid that the similarity between signatures depends on a specific hashing function. Subsequently signatures with two equal vectors in different bands are hashed differently. Also, to avoid hashing different portions of a signature in the same bucket it important to chose a great enough bucket.

In [121]:
import hashlib

def hash_band(list,seed,hash_bucket_size):
    m=hashlib.shake_256()
    m.update(bytearray(list))
    m.update(bytes(seed))
    hash_bucket_size = math.floor(hash_bucket_size/256)
    return int.from_bytes(m.digest(hash_bucket_size),'little')

def bucket_size(reviews_num,r):
    return math.factorial(reviews_num)/(math.factorial(reviews_num-r)*math.factorial(r))

hash_bucket_size = 256*2
review_hashed_signature_RDD = review_banded_signature_RDD.map(lambda s: (s[0],[hash_band(t[1],t[0],hash_bucket_size) for t in s[1]]))

Let's give a look at the new compact representation of a review contained in `review_banded_signature`.

In [ ]:
review_hashed_signature_RDD.first()

Now we search for candidate pairs among the reviews. We consider as possibly similar couples of reviews those that have Jaccard similarity at least $t$.

In [ ]:
def Jaccard(s:set,t:set):
    return len(s.intersection(t)) / len(s.union(t))

def RDD_similar_items(rdd,t):
    pairs_RDD = rdd.cartesian(rdd).filter(lambda s: s[0][0] is not s[1][0])
    return pairs_RDD.map(lambda s: ((s[0][0], s[0][1]),(s[1][0], s[1][1]))).filter(lambda s: Jaccard(set(s[1][0]),set(s[1][1]))>t)

candidate_review_pairs_RDD = RDD_similar_items(review_hashed_signature_RDD,t)

In [ ]:
candidate_review_pairs_RDD.first()